# Lesson 11 : Hosted Agents in Microsoft Foundry

When you deploy and run your generated agent's code for production, you can host your agent by yourself. By using Microsoft Agent Framework (MAF), especially, you can expose your agent via AG-UI-compliant FastAPI applications, OpenAI Responses, Telegram, A2A, MCP, and more.<br>
However, if you're working on Microsoft Foundry, you can host your agent by publishing into Microsoft Foundry as a **hosted agent**, with which you can obtain excellent architectural design.

It runs in a micro-VM sandbox (backed by the same infrastructure as **Azure Container Apps Sandboxes**), which provides CPU and memory allocation to run the agent.<br>
The hosted agent platform in Microsoft Foundry provides a fully managed runtime to deploy and operate it at enterprise scale - with security, isolation (hypervisor-isolated sandbox), and performance that production workloads demand. (Without changing any code, you can gain faster startup, zero idle cost, virtual network support, observability, built-in management functionalities, guardrails, Entra ID protection, and more.)

The code built by Microsoft Agent Framework, LangChain, and custom code can be run as a hosted agent. (Workflow agents in Microsoft Agent Framework can also work as hosted agents.)

Foundry hosted agents can be used and run in the following three ways :

- Assistive agents (chat agents) : act interactively on the user's behalf inside conversation (including voice live chat agent)
- Autonomous agents (routines) : act on their own behalf in the background by triggers (scheduling)
- Autopilot agents : act autonomously like humans (also long-running - such as, all day long) using their own identity and have user accounts with a productivity license granting them their own email, calendar, OneDrive, Microsoft Teams access

> Note : For running autopilot agents as Foundry hosted agents, use Agent 365 SDK. Please see [here](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/python/foundry-autopilot-agent) for this sample code.

There exist the following 2 types of deployment :

- Container-based hosted agent : The asset for deployment is a container, so the container image is built locally. (The runtime to build your container is required in your local environment.)
- Source-code-based hosted agent : The asset for deployment is a .zip file including code and it's built (installed) on remote server during deployment. (The source-code deployments simplifies CI/CD and keeps the deployment flexible - such as, integrating with GitHub Actions.)

In this exercise, we deploy and run an agent (written by Microsoft Agent Framework) as Foundry hosted agent in the following configuration :

- Deploy as code (Source-code-based hosted agent)
- Run as a simple chat agent (Assistive agent)

## 1. Prepare environment

To start this exercise, you should install ```azd``` command (Azure Developer CLI) on your working environment.

> Note : Here we use ```azd``` command to configure infrastructure, but you can also manually provision Azure infrastructure and deploy your hosted agent with Python SDK.

After installation, login to Azure in ```azd``` by running the following command.

```
azd auth login --use-device-code
```

Next, install ai agent extension (```azure.ai.agents```) in ```azd``` by running the following command.<br>
(Or install `microsoft.foundry` extension, which installs all extensions for Foundry AI agent development lifecycle.)

```
azd extension install azure.ai.agents
```

> Note : This notebook is experimented with azd version 1.37.2 and azd ai agent extension version 1.0.0-beta.23.

## 2. Prepare resources in Azure

Please create a **Foundry resource** (with a Foundry project) in the region where hosted agents are supported.<br>
For the supported regions by the hosted agent, please see [here](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents#region-availability).

Next, in Foundry Portal, deploy a **model**, which is supported in Foundry Responses API. (See [here](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/generate-responses).)

> Note : In this exercise, we prepare Foundry resource and model deployment in advance, but you can also create these resources in the following provisioning phase, without creating manually by yourself.

## 3. Prepare files

As I have mentioned above, we'll apply code-based deployment in this exercise.<br>
In order for provisioning as a hosted agent, therefore, you should prepare the following files in this example.

- Python source code (main.py)
- Python package list (requirements.txt)
- Agent configuration (azure.yaml) - this file will be automatically provisioned through `azd ai agent init`.

In this example, we'll store these files in the following structure.

```
agent-framework-responses/
├── azure.yaml
└── src/
    └── hosted-test-agent01/
        ├── main.py
        └── requirements.txt
```

First we create this folder (and sub-folders) as follows.

In [1]:
import os
os.makedirs("agent-framework-responses/src/hosted-test-agent01", exist_ok=True)

We prepare Python source code file (```main.py```).<br>
By specifying ```%%writefile``` directive in the first line in the notebook cell as follows, this code is stored as a file, not executed here. (The same applies to the following cells.)

Foundry hosted agents support 3 types of protocols - **open responses**, **invocations**, and **activity**.  
Since we'll use Open Responses protocol in this example, the agent server is abstracted by adapter function, ```ResponsesHostServer()```, as seen in the following code.

This wrapper class automates identity and monitoring as follows:

*1. Identidies*

It's worth noting that, unlike other examples run locally, the credential returned by ```DefaultAzureCredential()``` in hosted agent is **not the user's credential, but the app credential of the Agent identity** that identity holds "Foundry User" role for the Foundry project. If a user token is required in hosted agent (such as, in MCP tool calling), the Foundry platform internally restores the user token using the token passed by the caller. This fact is very important for handling credentials in your code - such as, in using [Foundry memory](./06_memory.ipynb), in tool calling, etc.<br>
Due to this mechanism, the caller (user) of this agent only needs "Foundry Agent Consumer" role in Foundry resource - not needing "Foundry User" role or "Contributor" role in Azure subscription.<br>
See [here](https://tsmatz.wordpress.com/2026/03/12/microsoft-agent-identity-for-developers/) for details about Agent identity.

*2. Monitoring*

When enabled tracing in your Foundry project, the platform automatically injects an Application Insights connection string into your agent container via environment variables, and the wrapper (`ResponsesHostServer`) integrates the tracing. So your agent built in Microsoft Agent Framework automatically emits traces (i.e., no additional code or packages are required, seen in [Lesson 2](./02_trace.ipynb)).<br>
I note that the internal hosting libraries used in hosted agents (```azure-ai-agentserver-responses``` and ```azure-ai-agentserver-invocations```) smoothly integrate using Microsoft OpenTelemetry distro library (```microsoft-opentelemetry```).

In [2]:
%%writefile agent-framework-responses/src/hosted-test-agent01/main.py
import os
from typing import Annotated
from pydantic import Field
from random import randint
from agent_framework.foundry import FoundryChatClient, ResponsesHostServer
from agent_framework import Agent, tool
from azure.identity import DefaultAzureCredential

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]}."

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."

if __name__ == "__main__":
    credential = DefaultAzureCredential()
    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        credential=credential,
    )
    agent = Agent(
        client=client,
        instructions="You are an agent about weather information.",
        tools=[get_weather, get_temperature],
        default_options={"store": False},
    )
    server = ResponsesHostServer(agent)
    server.run()

Writing agent-framework-responses/src/hosted-test-agent01/main.py


Next we prepare ```requirements.txt``` (a list of Python packages) which are installed on building code.

To use adapter for hosted agent (i.e., ```ResponsesHostServer()``` in above code), we should also install Python package ```agent-framework-foundry-hosting``` as follows.

In [3]:
%%writefile agent-framework-responses/src/hosted-test-agent01/requirements.txt
agent-framework
agent-framework-foundry-hosting

Writing agent-framework-responses/src/hosted-test-agent01/requirements.txt


## 4. Provision, deploy, and run

Now the assets are all ready.  
We now start to configure, provision Azure infrastructure, and deploy your hosted agent, with ```azd``` commands.

**All the following settings should be performed in console (terminal)**, not in Jupyter notebook. (Because Jupyter notebook cannot handle the interactive session.)

Before starting, let's change your working directory to the folder ```agent-framework-responses```. (Because the execution commands automatically detect the files in your current directory.)

```bash
cd agent-framework-responses
```

---

Run the following command to create the configuration (```azure.yaml```) and its environment settings.<br>
To get string for `--project-id` option, go to foundry project resource on [Azure Portal](https://portal.azure.com/), open "`Resource Management`" - "`properties`" in left-side navigation, and you then find it in "`Resource ID`". (Or **please change the following placeholders** manually.)

```
azd ai agent init --src ./src/hosted-test-agent01 \
  --agent-name hosted-test-agent01 \
  --project-id /subscriptions/[SUBSCRIPTION-ID]/resourceGroups/[RESOURCE-GROUP-NAME]/providers/Microsoft.CognitiveServices/accounts/[FOUNDRY-RESOURCE-NAME]/projects/[PROJECT-NAME]
```

When running this command, you will be asked for settings, and please specify the following settings.<br>

- project name : agent-framework-responses
- environment name : (set arbitrary name - such like "dev")
- deployment method (code or container) : source code (ZIP upload)
- runtime : Python 3.13
- entry point : main.py
- the method to resolve dependency : remote build
- deploy new model or use existing model : use an existing model deployment (and select the existing deployment)

All the configuration is saved in `azure.yaml` file and the environment settings are saved in `.azure/[env name]/.env` file.<br>
You can show all environment settings by running `azd env get-values`, and please make sure that both `FOUNDRY_PROJECT_ENDPOINT` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` exist in the settings, because these are used in source code (`main.py`). (If not, please add individual environment by running `azd env set` command.)

<blockquote>
Note : You can also manually create <code>azure.yaml</code>.<br>
In that case, you must also configure the environment settings manually, as described below.

```
azd env new dev
azd env set AZURE_SUBSCRIPTION_ID [SUBSCRIPTION-ID]
azd env set AZURE_RESOURCE_GROUP  [RESOURCE-GROUP-NAME]
azd env set AZURE_LOCATION [LOCATION-NAME] # such as, "japaneast"
azd env set USE_EXISTING_AI_PROJECT true
azd env set AZURE_AI_PROJECT_ID /subscriptions/[SUBSCRIPTION-ID]/resourceGroups/[RESOURCE-GROUP-NAME]/providers/Microsoft.CognitiveServices/accounts/[FOUNDRY-RESOURCE-NAME]/projects/[PROJECT-NAME]
azd env set FOUNDRY_PROJECT_ENDPOINT https://[FOUNDRY-RESOURCE-NAME].services.ai.azure.com/api/projects/[PROJECT-NAME]
azd env set AZURE_AI_MODEL_DEPLOYMENT_NAME [EXISTING-MODEL-DEPLOYMENT-NAME]
```
</blockquote>

---

**[Optional] This is not mandatory for running this exercise.**

When you run and test your AI agent locally.

To run your agent on your local environemnt, go to `agent-framework-responses` directory and run the following command. (This will automatically installs packages written in ```requirements.txt```.)

```
azd ai agent run
```

Open another console, go to `agent-framework-responses` directory, and send a message by running the following command.

```
azd ai agent invoke "Tell me the weather and temperature in Osaka today." --local
```

> Note : **Do not enable managed identity** in your working VM. If it's enabled, the managed identity will be used for your current credential (in ```DefaultAzureCredential``` in your code) and the execution will then fail.

---

Now let's provision the infrastructure by running the following command.

```bash
azd provision
```

---

Finally we deploy and start our hosted agent in Microsoft Foundry by running the following command.

```bash
azd deploy
```

> Note :  By running ```azd up```, provisioning (```azd provision```) and deployment (```azd deploy```) are both performed.<br>
> By running ```azd down```, it cleans up all resources in resource group.<br>
> By using ```azd up``` / ```azd down```, you can quickly build all resources and then clean up. (It's very useful in development.)

After the hosted agent is successfully deployed, you can see your agent running in Foundry Portal UI. (The hosted agent is labeled "```hosted```" as type.)

## 5. Consume your hosted agent

To consume your hosted agent, you can use Playground in Foundry Portal UI, or invoke API.  
Since the hosted agent in this example can talk with Open Responses protocol, you can use Azure AI Projects SDK (```azure-ai-projects```), OpenAI SDK, or raw REST in this case.  
The following code extracts generic ```OpenAI``` client by using Azure AI Projects SDK, and invokes the request with this client. (Before running this cell, **replace the placeholders with your Foundry settings**, in which your hosted agent is running.)

In [5]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from IPython.display import Markdown, display

AGENT_NAME = "hosted-test-agent01"
PROJECT_ENDPOINT = "https://[FOUNDRY-NAME].services.ai.azure.com/api/projects/[FOUNDRY-PROJECT-NAME]"

# initialize the client and retrieve the agent
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=AzureCliCredential(),
    allow_preview=True,
)

# get OpenAI client and send a message
# (it's OpenAI Responses API.)
openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)
conversation = openai_client.conversations.create()
response = openai_client.responses.create(
    conversation=conversation.id,
    input=[{"role": "user", "content": "Tell me the weather and temperature in Osaka today."}],
)

# show result
display(Markdown(response.output_text))

Today in Osaka, it’s sunny with a temperature of 15°C.

## 6. Use your hosted agent in Microsoft ecosystem (Publish as Microsoft 365 Copilot Agent)

Once it's deployed as hosted agent, your agent can also be consumed in Microsoft ecosystems.

For example, Microsoft 365 Copilot supports **Activity Protocol**, and your hosted agent can be exposed an agent that supports this protocol with one-click deployment experience in Foundry portal UI. (See below picture.)

![Publish to Microsoft 365](./assets/publish_m365.png)

## [Note] Other hosted agent integration features

In addition to identity management, monitoring, and M365 publishing capabilities (mentioned above), you can integrate with various features natively provided by Foundry hosted agent platform.<br>
Some of these are briefly introduced below.

### Retain user isolation in hosted agent

When handling stored data (persistence) using paths — such as, `$HOME` or `/files` — in Foundry hosted agents, you can ensure that users land in the same sandbox (user isolation) to maintain the server-side session, by the following steps.<br>
(Please note that this differs from preserving conversation history, which was explained in Lesson 3.)

1. A server-side session is created and `agent_session_id` is then returned — either by executing the initial turn or by explicitly creating a session using Foundry SDK.
2. Call the Foundry-hosted agent using above `agent_session_id` as follows. (This ensures that the user lands in the same sandbox.)

```
from agent_framework.foundry import FOUNDRY_HOSTED_AGENT_SESSION_ID_KEY

# set agent_session_id returned in the first call
session.state[
    FOUNDRY_HOSTED_AGENT_SESSION_ID_KEY
] = agent_session_id

# run agent retaining the server-side session
await agent.run("Tell me the weather in Osaka.", session=session)
```

See [official document](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/isolate-sessions-per-user) for details.

### Enable long-running recoverable execution

As you saw in Lesson 9 or Lesson 10, you may run long-running agent execution, while Foundry hosted agent platform also enables long-running resilent execution.<br>
You can connect workflow checkpoints and sessions to the resilient background in Foundry hosted platform as follows.

```
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.ai.agentserver.responses import ResponsesServerOptions
 
workflow_agent = workflow.as_agent(name="workflow-resilience-test")
 
server = ResponsesHostServer(
    workflow_agent,
    options=ResponsesServerOptions(resilient_background=True),
) 
server.run()
```

See [official document](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/deploy-resilient-agent) for details.